# Split&Splat trên LERF — chuẩn bị đầu vào cho bước lan truyền mask

README của Split&Splat nói bước `mask_propagation.py` cần, **cho mỗi scene**: ảnh, camera COLMAP, **depth**, và **point cloud 3D**. ScanNet có sẵn depth; **LERF thì không**. Notebook này tạo đúng những thứ đó theo bài báo
([arXiv 2602.03809](https://arxiv.org/html/2602.03809v1)): *"whenever depth information was not available in the dataset, the **Murre** approach (Guo et al., 2025) was employed for monocular depth estimation … due to its scale consistency with respect to the SfM point cloud (obtained via COLMAP) and consequently, with respect to Gaussian splatting"*, và point cloud dày đến từ *"depth-regularized Gaussian splatting, which will generate a dense point cloud of the scene"*.

```
ảnh LERF ─► COLMAP (convert.py của tác giả) ─► sparse/0 (PINHOLE) + images/
                         │
                         ├─► Murre (SfM-guided depth) ─► depth/<tên>_pred.npy
                         │
                         └─► 3DGS có depth regularization (train.py của tác giả, -d depths) ─► point cloud dày ─► sparse/0/points3D.ply
                                                                         │
                                          ◄───────────── data/<scene>/{images, sparse, depth} ─► notebook lan truyền mask
```

### Những chỗ tác giả **không nêu rõ** (tôi buộc phải chọn — bạn chỉnh được ở ô tham số)

| Chỗ | Lựa chọn trong notebook | Vì sao |
|---|---|---|
| Camera COLMAP | `PINHOLE` (cờ `--camera` có sẵn trong `convert.py`) | Ảnh LERF đã được undistort sẵn (transforms.json không có k1,k2,p1,p2). PINHOLE giữ nguyên kích thước ảnh nên **mask SAM 2 đã chạy vẫn khớp**; mặc định `OPENCV` sẽ ước lượng méo rồi cắt/xoay ảnh làm lệch mask |
| COLMAP | CPU nếu bản `colmap` của apt không có CUDA | `convert.py` dùng `exhaustive_matcher`; chậm hơn GPU nhưng cùng thuật toán |
| Tham số Murre | `--processing_res 0 --max_depth 10.0 --denoise_steps 10 --ensemble_size 5` | đúng lệnh trong README của Murre cho cảnh trong nhà |
| Depth → 3DGS | đổi depth Murre thành *inverse depth* PNG 16-bit rồi chạy `utils/make_depth_scale.py` của tác giả | `train.py` của tác giả đọc `depths/<tên>.png` theo quy ước 3DGS (`/2**16`) và tự khớp scale/offset từng ảnh với COLMAP |
| Số vòng lặp 3DGS | `10000` | bài báo chỉ nêu "10k iterations" cho LERF ở bước dựng từng vật, không nêu cho point cloud dày |
| `sparse/0/points3D.ply` | `point_cloud.ply` của 3DGS sau khi train | bài báo: point cloud dày lấy từ 3DGS; `mask_propagation.py` đọc đúng đường dẫn này |

## 0. Kiểm tra GPU

In [2]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Chưa có GPU: Runtime ▸ Change runtime type ▸ chọn GPU (khuyên dùng L4) rồi chạy lại."
print("GPU:", gpu)

GPU: NVIDIA L4, 23034 MiB, 8.9


## 1. Tham số *(sửa ở đây)*

In [3]:
#@title 1. Tham số { display-mode: "form" }

#@markdown ### Dữ liệu LERF (zip trên Drive)
LERF_GDRIVE_IDS = "1t08_1xV1h5pCdU6-oKRP49veUjxUSM_8, 1LqmUsixkDkw7GIWQX0ZmJxXjFdMMlPjc"  #@param {type:"string"}
LERF_ZIP_DIR    = "/content/lerf_zips"  #@param {type:"string"}
SCENES = "all"  #@param {type:"string"}
#@markdown Hai ID mặc định là `figurines.zip` và `teatime.zip`. `SCENES`: `all` hoặc liệt kê tên (vd `teatime`).

#@markdown ### Mã nguồn
REPO_URL = "https://github.com/LTTM/Split_and_Splat.git"  #@param {type:"string"}
REPO_REF = "9b7ba906511f75b87c438b4cf30ae3d1d5302f61"  #@param {type:"string"}
MURRE_URL = "https://github.com/zju3dv/Murre.git"  #@param {type:"string"}
MURRE_REF = "main"  #@param {type:"string"}
MURRE_CKPT_GDRIVE_ID = "11hyjtfZhDcVK2l0tAf9OhxH_Uu1JuvFY"  #@param {type:"string"}
CKPT_SEED_GDRIVE_ID = "1cPFOvfkLvqKY09yo098ndka_PFfnhYzq"  #@param {type:"string"}
#@markdown Hai ID trên là file zip bạn tự upload (Murre-ckpt.zip và ckpt-...zip = checkpoint depth/COLMAP của lần chạy trước). Notebook tải về `/content` bằng gdown. File trên Drive phải để chế độ **Anyone with the link**. Để trống `CKPT_SEED_GDRIVE_ID` nếu chạy từ đầu.
MURRE_CKPT_LOCAL = ""  #@param {type:"string"}

#@markdown ### Thư mục dữ liệu trên Drive (checkpoint Murre, zip LERF, kết quả đều dùng chung thư mục này)
DRIVE_DIR = ""  #@param {type:"string"}
MURRE_CKPT_EVERY = 50  #@param {type:"integer"}
AUTO_DOWNLOAD_CKPT = True  #@param {type:"boolean"}
#@markdown **Chạy qua VS Code + extension Colab:** `files.download` không hoạt động. Muốn dùng checkpoint ở máy: chuột phải thư mục `Data/ckpt-.../` (hoặc zip) trong Explorer ▸ **Upload to Colab** (vào `/content`), notebook tự nhận. Lấy kết quả về máy bằng **Colab: Mount Server To Workspace...** (bật experimental trong Settings) hoặc view **Contents**.
#@markdown Checkpoint trên Drive (thư mục `DRIVE_DIR/ckpt/<scene>/`): sau COLMAP, sau mỗi `MURRE_CKPT_EVERY` ảnh depth của Murre, và sau 3DGS. Phiên mới tự khôi phục và làm tiếp từ chỗ dở. Mỗi lần chia nhóm Murre phải nạp lại model (~1 phút), nên đừng đặt số quá nhỏ.
#@markdown Tự tìm trong thư mục này: `Murre-ckpt.zip`, `figurines.zip`, `teatime.zip` (có thì không tải từ Drive công khai nữa, tránh lỗi quota). `<scene>_prep.zip` được lưu về đây sau mỗi scene. Để trống = tắt, dùng cách cũ (gdown + tải về máy).
#@markdown `MURRE_CKPT_LOCAL`: đường dẫn checkpoint Murre đã có sẵn (thư mục hoặc .zip), vd `/content/drive/MyDrive/murre_ckpt.zip` hoặc file upload vào `/content/`. Dùng khi Drive báo *"Too many users have viewed or downloaded this file"* (xem ô checkpoint bên dưới). Để trống = tải bằng gdown.

#@markdown ### COLMAP
CAMERA_MODEL = "PINHOLE"  #@param ["PINHOLE", "OPENCV", "SIMPLE_PINHOLE"] {allow-input: true}
COLMAP_GPU   = "auto"  #@param ["auto", "no"]
#@markdown `PINHOLE` giữ nguyên kích thước ảnh (khớp mask SAM 2 đã có). Đổi sang `OPENCV` là đúng mặc định của `convert.py` nhưng ảnh bị undistort/cắt → mask phải chạy lại trên ảnh mới.

#@markdown ### Murre (depth)
MURRE_PROCESSING_RES = 0    #@param {type:"integer"}
MURRE_MAX_DEPTH      = 10.0 #@param {type:"number"}
MURRE_DENOISE_STEPS  = 10   #@param {type:"integer"}
MURRE_ENSEMBLE_SIZE  = 5    #@param {type:"integer"}
MURRE_ERR_THR   = ""  #@param {type:"string"}
MURRE_NVIEWS_THR = ""  #@param {type:"string"}
#@markdown `MURRE_MAX_DEPTH`: 10.0 cho cảnh trong nhà (README Murre). Đơn vị depth là đơn vị của COLMAP (tuỳ ý) — nếu notebook cảnh báo depth COLMAP vượt ngưỡng này thì tăng lên.

#@markdown ### 3DGS có depth regularization (dựng point cloud dày)
GS_ITERATIONS = 10000  #@param {type:"integer"}

#@markdown ### Tải kết quả về máy
DOWNLOAD_WHEN = "after_each_scene"  #@param ["after_each_scene", "at_end", "never"]
ZIP_PARTS = 1  #@param {type:"integer"}
#@markdown Kết quả mỗi scene: `<scene>_prep.zip` gồm `images/`, `sparse/0/`, `depth/`. Đưa file này (hoặc ID Drive của nó) vào notebook lan truyền mask.

## 2. Môi trường

CUDA 12.x + Python 3.12 (torch 2.5.1) cho COLMAP/3DGS giống `environment.yml`; Python 3.10 + torch 2.0.1 (CUDA 11.8) riêng cho **Murre** (đúng yêu cầu repo Murre).

In [4]:
import os, sys, re, glob, json, shutil, subprocess, time, zipfile
from pathlib import Path

def try_download(path):
    """Colab web: files.download tải về trình duyệt. VS Code + extension Colab: files.download KHÔNG làm gì → in đường dẫn để lấy file bằng
    'Colab: Mount Server To Workspace...' (thử nghiệm) hoặc view 'Contents' của extension."""
    p = Path(path)
    print(f"  📁 {p}  ({p.stat().st_size / 1e6:.0f} MB) nằm trên máy Colab")
    print("     VS Code: mở lệnh 'Colab: Mount Server To Workspace...' hoặc view 'Contents' rồi chép file này về máy.")
    try:
        from google.colab import files
        files.download(str(p))
    except Exception as e:
        print(f"     (không tự tải được: {e})")

def ensure_drive():
    if not str(globals().get("DRIVE_DIR", "")).strip():
        return                      # không dùng Drive: mọi thứ chạy trên /content
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

def resolve_drive_dir(dir_hint, marker="Murre-ckpt.zip", max_depth=6):
    """Dùng DRIVE_DIR nếu có đủ marker; nếu không, tìm thư mục chứa marker trong MyDrive (vd thư mục chia sẻ bạn tự upload)."""
    ensure_drive()
    hint = Path(dir_hint)
    if (hint / marker).exists():
        return str(hint)
    base = Path("/content/drive/MyDrive")
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        dirs[:] = [d for d in dirs if d not in (".Trash", ".shortcut-targets-by-id") and not d.startswith(".")] if depth < max_depth else []
        if marker in files:
            print(f"Tìm thấy {marker} tại {root} → dùng làm DRIVE_DIR")
            return root
    print(f"⚠️ Không thấy {marker} trong Drive; giữ DRIVE_DIR = {dir_hint}")
    return dir_hint

_NOISY = re.compile(r"Diffusion denoising|Inference batches|feature_extraction\.cc:\d+\]\s+(Name|Dimensions|Camera|Focal Length|Features|SKIP)|^\s*$")
_DEPTH_N = re.compile(r"Estimating depth: (\d+)it")
_nproc = {"n": 0}
def _show(line):
    """Chỉ in dòng quan trọng: bỏ thanh tiến độ tqdm và log từng ảnh của COLMAP, giữ 1 dòng mỗi 10 ảnh."""
    if _NOISY.search(line):
        return False
    if "Processed file" in line:
        _nproc["n"] += 1
        return _nproc["n"] % 25 == 1
    m = _DEPTH_N.search(line)
    if m:
        return int(m.group(1)) % 10 == 0
    return True

def sh(cmd, cwd=None, env=None, check=True, quiet=False):
    """Chạy lệnh bash và in log ngay khi có (không đợi lệnh xong). Lỗi thì nêu 25 dòng cuối."""
    print("$", cmd if len(cmd) < 400 else cmd[:400] + " …")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not quiet and _show(line):
            print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {rc}). 25 dòng cuối:\n" + "".join(tail[-25:]))
    return rc

def _nvcc_version(home):
    try:
        out = subprocess.run([f"{home}/bin/nvcc", "--version"], capture_output=True, text=True).stdout
        m = re.search(r"release (\d+)\.(\d+)", out)
        return (int(m[1]), int(m[2])) if m else None
    except Exception:
        return None

def find_or_install_cuda12():
    """torch 2.5.1 (CUDA 12.4) chỉ build được extension CUDA nếu nvcc cùng họ 12.x (Colab mới có thể chỉ có nvcc 13)."""
    for home in sorted(glob.glob("/usr/local/cuda-12.*")) + ["/usr/local/cuda"]:
        v = _nvcc_version(home)
        if v and v[0] == 12:
            print("Dùng CUDA", v, "tại", home)
            return home
    osr = open("/etc/os-release").read()
    m = re.search(r'VERSION_ID="(\d+)\.(\d+)"', osr)
    distro = f"ubuntu{m[1]}{m[2]}"
    print(f"Chưa có nvcc 12.x → cài CUDA toolkit 12.x từ kho NVIDIA ({distro}), mất vài phút …")
    sh(f"wget -q https://developer.download.nvidia.com/compute/cuda/repos/{distro}/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb"
       f" && dpkg -i /tmp/cuda-keyring.deb && apt-get update -qq")
    for v in ("12-4", "12-6", "12-8"):     # ưu tiên đúng 12.4 như môi trường tác giả; bản mới hơn chỉ khác phiên bản phụ
        if sh(f"apt-get install -y -qq cuda-toolkit-{v}", check=False) == 0:
            home = f"/usr/local/cuda-{v.replace('-', '.')}"
            print("Đã cài", home)
            return home
    raise RuntimeError("Không cài được CUDA toolkit 12.x bằng apt.")

CUDA_HOME = find_or_install_cuda12()
COMPUTE_CAP = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip().splitlines()[0]
print("CUDA_HOME =", CUDA_HOME, "| compute capability", COMPUTE_CAP)

Chưa có nvcc 12.x → cài CUDA toolkit 12.x từ kho NVIDIA (ubuntu2404), mất vài phút …
$ wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb && dpkg -i /tmp/cuda-keyring.deb && apt-get update -qq
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack /tmp/cuda-keyring.deb ...
Unpacking cuda-keyring (1.1-1) over (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
$ apt-get install -y -qq cuda-toolkit-12-4
E: Unable to locate package cuda-toolkit-12-4
$ apt-get install -y -qq cuda-toolkit-12-6
Selecting previously unselected package at-spi2-common.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database

In [5]:
def make_env(venv, extra_pythonpath=""):
    """Biến môi trường để chạy script trong venv với extension CUDA (README bước 6: LD_LIBRARY_PATH của torch)."""
    env = os.environ.copy()
    env["CUDA_HOME"] = CUDA_HOME
    env["PATH"] = f"{CUDA_HOME}/bin:{venv}/bin:" + env["PATH"]
    env["TORCH_CUDA_ARCH_LIST"] = COMPUTE_CAP
    env["MAX_JOBS"] = "4"
    env["PYTHONUNBUFFERED"] = "1"
    try:
        torch_lib = subprocess.check_output([f"{venv}/bin/python", "-c",
                                             "import torch, os; print(os.path.join(os.path.dirname(torch.__file__), 'lib'))"],
                                            text=True).strip()
        env["LD_LIBRARY_PATH"] = f"{torch_lib}:{CUDA_HOME}/lib64:" + env.get("LD_LIBRARY_PATH", "")
    except Exception:
        pass
    if extra_pythonpath:
        env["PYTHONPATH"] = extra_pythonpath + ":" + env.get("PYTHONPATH", "")
    return env

sh("apt-get install -y -qq libgl1 libglib2.0-0 libgomp1", check=False)
sh("pip -q install uv gdown")

# --- venv 3.12: COLMAP tools + 3DGS (train.py của tác giả) ---
VENV = Path("/content/venv312"); VPY = VENV / "bin" / "python"
if not VPY.exists():
    sh(f"uv venv --python 3.12 {VENV}")
def vpip(args, env=None):
    sh(f"uv pip install --python {VPY} {args}", env=env)
vpip("torch==2.5.1 torchvision==0.20.1")
vpip("numpy==2.1.3 pillow==11.1.0 scipy==1.15.2 scikit-learn==1.6.1 matplotlib==3.10.1 opencv-python==4.11.0.86 "
     "open3d==0.19.0 pycolmap==3.11.1 joblib==1.4.2 pandas==2.2.3 plyfile tqdm")
vpip("setuptools wheel ninja")
sh(f"{VPY} -c \"import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPU:', torch.cuda.is_available())\"")

$ apt-get install -y -qq libgl1 libglib2.0-0 libgomp1
$ pip -q install uv gdown
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 122.2 MB/s eta 0:00:00
$ uv venv --python 3.12 /content/venv312
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: venv312
Activate with: source venv312/bin/activate
$ uv pip install --python /content/venv312/bin/python torch==2.5.1 torchvision==0.20.1
Using Python 3.12.3 environment at: venv312
Resolved 26 packages in 432ms
Prepared 26 packages in 25.27s
Installed 26 packages in 174ms
 + filelock==4.0.12
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.7
 + numpy==2.5.3
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.3
 + nvidia-curand-cu12==10.3.5.147
 + nvidia-cusolver-cu12==11.6.1.9
 + nvidia-cusparse-cu12==12.3.1.170

0

In [6]:
# --- venv 3.10: Murre (README của Murre: python 3.10, torch 2.0.1, CUDA 11.8) ---
VENV_M = Path("/content/venv310"); VPY_M = VENV_M / "bin" / "python"
if not VPY_M.exists():
    sh(f"uv venv --python 3.10 {VENV_M}")
sh(f"uv pip install --python {VPY_M} 'torch==2.0.1+cu118' 'torchvision==0.15.2+cu118' "
   f"--extra-index-url https://download.pytorch.org/whl/cu118 --index-strategy unsafe-best-match")
# requirements.txt của Murre (bỏ dòng `pip==...`)
sh(f"uv pip install --python {VPY_M} numpy==1.26.0 opencv-python==4.8.0.76 tqdm==4.66.1 trimesh==4.0.2 pillow==11.1.0 "
   f"matplotlib==3.8.3 scikit-learn==1.4.1.post1 diffusers==0.27.2 transformers==4.39.1 scipy==1.11.2 "
   f"typing-extensions==4.10.0 huggingface_hub==0.25.0 open3d==0.18.0")
sh(f"{VPY_M} -c \"import torch, diffusers; print('Murre env: torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '| diffusers', diffusers.__version__)\"")

$ uv venv --python 3.10 /content/venv310
Using CPython 3.10.22
Creating virtual environment at: venv310
Activate with: source venv310/bin/activate
$ uv pip install --python /content/venv310/bin/python 'torch==2.0.1+cu118' 'torchvision==0.15.2+cu118' --extra-index-url https://download.pytorch.org/whl/cu118 --index-strategy unsafe-best-match
Using Python 3.10.22 environment at: venv310
Resolved 19 packages in 818ms
Prepared 14 packages in 18.63s
Installed 19 packages in 211ms
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + cmake==4.4.4
 + filelock==4.0.12
 + idna==3.20
 + jinja2==3.1.6
 + lit==23.1.3
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.34.2
 + sympy==1.14.0
 + torch==2.0.1+cu118
 + torchvision==0.15.2+cu118
 + triton==2.0.0
 + typing-extensions==4.16.0
 + urllib3==2.8.0
$ uv pip install --python /content/venv310/bin/python numpy==1.26.0 opencv-python==4.8.0.76 tqdm==4.66.1 trimesh==4.0.2 pillow==11.1.0 matplotlib

0

## 3. Mã nguồn: Split&Splat, Murre, bản COLMAP; build extension CUDA của 3DGS

In [7]:
REPO = Path("/content/Split_and_Splat")
if not REPO.exists():
    sh(f"git clone --recursive {REPO_URL} {REPO}")
sh(f"git -C {REPO} checkout -q {REPO_REF} && git -C {REPO} submodule update --init --recursive -q")
print("Split&Splat:", subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True).strip())

MURRE = Path("/content/Murre")
if not MURRE.exists():
    sh(f"git clone {MURRE_URL} {MURRE}")
sh(f"git -C {MURRE} checkout -q {MURRE_REF}")
MURRE_COMMIT = subprocess.check_output(["git", "-C", str(MURRE), "rev-parse", "HEAD"], text=True).strip()
print("Murre:", MURRE_COMMIT)

if DRIVE_DIR.strip():
    DRIVE_DIR = resolve_drive_dir(DRIVE_DIR)
    print("DRIVE_DIR =", DRIVE_DIR)

# checkpoint Murre (Drive). Có thể là zip hoặc thư mục diffusers (có model_index.json).
CKPT_DIR = Path("/content/murre_ckpt")
def find_ckpt_root(base):
    cands = [p.parent for p in Path(base).rglob("model_index.json")]
    return cands[0] if cands else None
def _unpack_zips(base):
    for z in Path(base).rglob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.parent)

if find_ckpt_root(CKPT_DIR) is None:
    CKPT_DIR.mkdir(exist_ok=True)
    ckpt_local = MURRE_CKPT_LOCAL.strip()
    if not ckpt_local and DRIVE_DIR.strip():
        ensure_drive()
        cand = Path(DRIVE_DIR) / "Murre-ckpt.zip"
        if cand.exists():
            ckpt_local = str(cand)
            print("Dùng checkpoint Murre từ Drive:", cand)
    if ckpt_local:
        local = Path(ckpt_local)
        if str(local).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
            from google.colab import drive
            drive.mount("/content/drive")
        assert local.exists(), f"MURRE_CKPT_LOCAL không tồn tại: {local}"
        if local.is_dir():
            sh(f"cp -r {local}/. {CKPT_DIR}/")
        else:
            shutil.copy(local, CKPT_DIR / local.name)
    else:
        rc = sh(f"gdown {MURRE_CKPT_GDRIVE_ID} -O {CKPT_DIR}/", check=False)
        if rc != 0:      # thử lại dạng folder/fuzzy trước khi bỏ cuộc
            rc = sh(f"gdown --fuzzy 'https://drive.google.com/file/d/{MURRE_CKPT_GDRIVE_ID}/view' -O {CKPT_DIR}/", check=False)
        if rc != 0:
            raise RuntimeError(
                "Không tải được checkpoint Murre bằng gdown (quota hoặc chưa chia sẻ). Cách xử lý:\n"
                f"  1. Tải checkpoint bằng trình duyệt: https://drive.google.com/file/d/{MURRE_CKPT_GDRIVE_ID}/view\n"
                "  2. Upload lên Drive của bạn (hoặc vào /content/ bằng panel Files).\n"
                "  3. Đặt file ở chế độ 'Anyone with the link' rồi điền MURRE_CKPT_GDRIVE_ID, hoặc điền MURRE_CKPT_LOCAL (đường dẫn trên Colab), chạy lại ô này.")
    _unpack_zips(CKPT_DIR)
MURRE_CKPT = find_ckpt_root(CKPT_DIR)
assert MURRE_CKPT is not None, f"Không thấy model_index.json trong {CKPT_DIR}: xem lại checkpoint Murre (gdown id {MURRE_CKPT_GDRIVE_ID})."
print("Checkpoint Murre:", MURRE_CKPT)

# COLMAP (convert.py của tác giả gọi lệnh `colmap`). Bản apt KHÔNG có CUDA → build từ source với CUDA để chạy GPU.
COLMAP_TAG = "3.11.1"
def colmap_has_cuda():
    if shutil.which("colmap") is None:
        return False
    r = subprocess.run(["colmap", "-h"], capture_output=True, text=True, env={**os.environ, "QT_QPA_PLATFORM": "offscreen"})
    t = (r.stdout + r.stderr)
    return ("with CUDA" in t) and ("without CUDA" not in t)

if COLMAP_GPU == "auto" and not colmap_has_cuda():
    try:
        sh("apt-get install -y -qq git cmake ninja-build build-essential libboost-program-options-dev libboost-graph-dev "
           "libboost-system-dev libeigen3-dev libfreeimage-dev libmetis-dev libgoogle-glog-dev libgtest-dev libgmock-dev "
           "libsqlite3-dev libglew-dev libcgal-dev libceres-dev libcurl4-openssl-dev libssl-dev libflann-dev libglapi-mesa libopengl-dev libgl1-mesa-dev")
        sh(f"rm -rf /content/colmap_src && git clone --depth 1 --branch {COLMAP_TAG} https://github.com/colmap/colmap.git /content/colmap_src")
        arch = COMPUTE_CAP.replace(".", "")
        sh(f"cd /content/colmap_src && cmake -B build -GNinja -DCMAKE_BUILD_TYPE=Release -DGUI_ENABLED=OFF "
           f"-DCUDA_ENABLED=ON -DCMAKE_CUDA_COMPILER={CUDA_HOME}/bin/nvcc -DCMAKE_CUDA_ARCHITECTURES={arch} "
           f"-DTESTS_ENABLED=OFF "
           f"-DCMAKE_EXE_LINKER_FLAGS='-Wl,--no-as-needed /usr/lib/x86_64-linux-gnu/libglapi.so.0 -Wl,--allow-shlib-undefined' "
           f"&& cmake --build build -j$(nproc) && cmake --install build")
    except Exception as e:
        print("Build COLMAP CUDA lỗi → dùng bản apt (CPU):", str(e)[-600:])
if shutil.which("colmap") is None:
    sh("apt-get install -y -qq colmap")
colmap_ver = subprocess.run(["colmap", "-h"], capture_output=True, text=True, env={**os.environ, "QT_QPA_PLATFORM": "offscreen"})
head = (colmap_ver.stdout + colmap_ver.stderr).strip().splitlines()[:3]
print("COLMAP:", " | ".join(head))
COLMAP_HAS_CUDA = ("with CUDA" in " ".join(head)) and ("without CUDA" not in " ".join(head))
USE_COLMAP_GPU = COLMAP_HAS_CUDA and COLMAP_GPU == "auto"
print("COLMAP dùng GPU:", USE_COLMAP_GPU, "(bản apt thường không có CUDA → chạy CPU, chậm hơn nhưng cùng thuật toán)")

# README mục 4: extension CUDA của 3DGS (train.py cần)
ENV = make_env(VENV)
for sub in ("diff-gaussian-rasterization", "simple-knn", "fused-ssim"):
    sh(f"uv pip install --python {VPY} {REPO}/submodules/{sub} --no-build-isolation", env=ENV)
ENV = make_env(VENV, extra_pythonpath=str(REPO))
sh(f"{VPY} -c \"import diff_gaussian_rasterization, simple_knn, fused_ssim; print('3DGS CUDA extensions OK')\"", cwd=REPO, env=ENV)

$ git clone --recursive https://github.com/LTTM/Split_and_Splat.git /content/Split_and_Splat
Cloning into '/content/Split_and_Splat'...
$ git -C /content/Split_and_Splat checkout -q 9b7ba906511f75b87c438b4cf30ae3d1d5302f61 && git -C /content/Split_and_Splat submodule update --init --recursive -q
Split&Splat: 9b7ba90 initial commit
$ git clone https://github.com/zju3dv/Murre.git /content/Murre
Cloning into '/content/Murre'...
$ git -C /content/Murre checkout -q main
Murre: 1c759de92e9f323e4565c7506121df47c1d457ce
$ gdown 11hyjtfZhDcVK2l0tAf9OhxH_Uu1JuvFY -O /content/murre_ckpt/
Downloading...
From (original): https://drive.google.com/uc?id=11hyjtfZhDcVK2l0tAf9OhxH_Uu1JuvFY
From (redirected): https://drive.google.com/uc?id=11hyjtfZhDcVK2l0tAf9OhxH_Uu1JuvFY&confirm=t&uuid=549b6b44-b918-42c0-964e-3bbfb0574ea1
To: /content/murre_ckpt/Murre-ckpt.zip
  0%|          | 0.00/4.79G [00:00<?, ?B/s]
  0%|          | 1.05M/4.79G [00:00<08:34, 9.30MB/s]
  0%|          | 4.72M/4.79G [00:00<05:38, 14.1

0

## 4. Dữ liệu LERF

Giải nén zip rồi tìm scene (thư mục `images/` có ảnh). Ảnh được đổi tên theo **đúng quy tắc của `sam2/auto_seg.py`** (`frame_00001.jpg` → `00001.jpg`: `new_name.replace("frame_", "")`), để tên ảnh COLMAP, tên mask Stage 1 và tên depth đều trùng nhau.

In [8]:
LERF_ZIP_DIR = Path(LERF_ZIP_DIR); LERF_ZIP_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_DIR.strip():                      # ưu tiên zip LERF nằm trong thư mục Drive của bạn (đúng tên figurines.zip / teatime.zip)
    ensure_drive()
    for name in ("figurines.zip", "teatime.zip"):
        src = Path(DRIVE_DIR) / name
        if src.exists() and not (LERF_ZIP_DIR / name).exists():
            print("Chép từ Drive:", src)
            shutil.copy(src, LERF_ZIP_DIR / name)
if LERF_GDRIVE_IDS.strip() and len(list(LERF_ZIP_DIR.glob("*.zip"))) < len([x for x in LERF_GDRIVE_IDS.replace("\n", ",").split(",") if x.strip()]):
    for fid in [x.strip() for x in LERF_GDRIVE_IDS.replace("\n", ",").split(",") if x.strip()]:
        sh(f"gdown {fid} -O {LERF_ZIP_DIR}/")
RAW = Path("/content/lerf_raw"); RAW.mkdir(exist_ok=True)
for zp in sorted(LERF_ZIP_DIR.glob("*.zip")):
    dest = RAW / zp.stem
    if dest.is_dir() and any(dest.iterdir()):
        continue
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(dest, [m for m in zf.namelist() if "__MACOSX" not in m and not m.endswith(".DS_Store")])
    print("giải nén", zp.name)

IMG_EXTS = (".jpg", ".jpeg", ".png")
found = {}
for d in sorted(RAW.rglob("images")):
    if d.is_dir() and "__MACOSX" not in d.parts and any(p.suffix.lower() in IMG_EXTS for p in d.iterdir()):
        found.setdefault(d.parent.name, d)
want = None if SCENES.strip().lower() == "all" else {s.strip() for s in SCENES.split(",") if s.strip()}
LERF_SCENES = {k: v for k, v in found.items() if want is None or k in want}
assert LERF_SCENES, f"Không tìm thấy scene nào trong {RAW} (tìm thư mục 'images' có ảnh)."
for k, v in LERF_SCENES.items():
    print(f"scene {k}: {sum(1 for p in v.iterdir() if p.suffix.lower() in IMG_EXTS)} ảnh  ← {v}")

def stage_images(scene, src_dir):
    """data/<scene>/input/<tên>.jpg  với tên bỏ tiền tố 'frame_' (quy tắc của auto_seg.py)."""
    dst = REPO / "data" / scene / "input"
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for p in sorted(src_dir.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            shutil.copy2(p, dst / p.name.replace("frame_", ""))
            n += 1
    return n

$ gdown 1t08_1xV1h5pCdU6-oKRP49veUjxUSM_8 -O /content/lerf_zips/
Downloading...
From (original): https://drive.google.com/uc?id=1t08_1xV1h5pCdU6-oKRP49veUjxUSM_8
From (redirected): https://drive.google.com/uc?id=1t08_1xV1h5pCdU6-oKRP49veUjxUSM_8&confirm=t&uuid=1efd25ea-1023-403b-b5d1-a29f3044fc03
To: /content/lerf_zips/figurines.zip
  0%|          | 0.00/58.0M [00:00<?, ?B/s]
  2%|▏         | 1.05M/58.0M [00:00<00:05, 9.52MB/s]
 15%|█▌        | 8.91M/58.0M [00:00<00:01, 36.9MB/s]
 30%|██▉       | 17.3M/58.0M [00:00<00:00, 54.7MB/s]
 41%|████      | 23.6M/58.0M [00:00<00:00, 48.8MB/s]
 62%|██████▏   | 36.2M/58.0M [00:00<00:00, 54.8MB/s]
 72%|███████▏  | 41.9M/58.0M [00:00<00:00, 52.7MB/s]
 91%|█████████▏| 53.0M/58.0M [00:01<00:00, 54.2MB/s]
100%|██████████| 58.0M/58.0M [00:01<00:00, 52.2MB/s]
$ gdown 1LqmUsixkDkw7GIWQX0ZmJxXjFdMMlPjc -O /content/lerf_zips/
Downloading...
From (original): https://drive.google.com/uc?id=1LqmUsixkDkw7GIWQX0ZmJxXjFdMMlPjc
From (redirected): https://drive.go

## 5. Các bước xử lý từng scene

Mỗi bước tự bỏ qua nếu kết quả đã có (chạy lại notebook không làm lại từ đầu).

1. **COLMAP** — `python convert.py -s data/<scene> --camera PINHOLE [--no_gpu]` (script của tác giả: feature_extractor → exhaustive_matcher → mapper → image_undistorter).
2. **Murre** — `sfm_depth/get_sfm_depth.py` rồi `run.py` (đúng lệnh trong README của Murre) → `depth_npy/<tên>_pred.npy`.
3. **3DGS có depth regularization** — depth → inverse-depth PNG → `utils/make_depth_scale.py` → `train.py -d depths`.
4. **Hoàn thiện** — `sparse/0/points3D.ply` = point cloud 3DGS; ảnh `.jpg` → `.JPEG` (đúng thao tác `auto_seg.py` làm khi đổi tên ảnh, và là tên mà `mask_propagation.py` đọc).

In [9]:
# ===== Checkpoint trên Drive: DRIVE_DIR/ckpt/<scene>/ =====
CKPT_LOCAL_DIR = Path("/content/ckpt")      # DRIVE_DIR trống → checkpoint ghi trên đĩa của Colab (mất khi hết phiên, nên có thêm bước tải về máy)

def ckpt_root(scene):
    return (Path(DRIVE_DIR) / "ckpt" / scene) if DRIVE_DIR.strip() else (CKPT_LOCAL_DIR / scene)

SEED_DIR = Path("/content/ckpt_seed")      # nơi giải nén zip checkpoint tải từ Drive (ckpt-*.zip), chỉ để ĐỌC, không ghi thêm lên Drive

SEED_ZIPS_DIR = Path("/content/seed_zips")

def seed_ckpt_zips():
    """Giải nén checkpoint 'hạt giống' (zip tải nguyên thư mục ckpt/) vào /content/ckpt_seed.
    Nguồn: file gdown từ CKPT_SEED_GDRIVE_ID, hoặc DRIVE_DIR/ckpt-*.zip nếu có dùng Drive."""
    if str(CKPT_SEED_GDRIVE_ID).strip() and not list(SEED_ZIPS_DIR.glob("*.zip")) and not list(Path("/content").glob("ckpt-*.zip")) \
            and not list(Path("/content").glob("*/ckpt/*")) and not list(Path("/content").glob("*/*/ckpt/*")):
        SEED_ZIPS_DIR.mkdir(parents=True, exist_ok=True)
        rc = sh(f"gdown {CKPT_SEED_GDRIVE_ID} -O {SEED_ZIPS_DIR}/", check=False)
        if rc != 0:
            print("  ⚠️ không tải được checkpoint hạt giống (kiểm tra CKPT_SEED_GDRIVE_ID và chế độ chia sẻ 'Anyone with the link') → chạy từ đầu")
    zips = sorted(SEED_ZIPS_DIR.glob("*.zip")) + sorted(Path("/content").glob("ckpt-*.zip"))   # zip tải bằng gdown hoặc upload tay
    if DRIVE_DIR.strip():
        ensure_drive(); zips += sorted(Path(DRIVE_DIR).glob("ckpt-*.zip"))
    for z in zips:
        mark = SEED_DIR / f".done_{z.name}"
        if mark.exists(): continue
        if not verify_zip(z):
            print(f"  ⚠️ {z.name} không đọc được → bỏ qua"); continue
        SEED_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(SEED_DIR)
        mark.write_text("ok")
        print(f"  ♻️ giải nén checkpoint {z.name} → {SEED_DIR}")

def ckpt_roots(scene):
    """Thư mục checkpoint của scene để ĐỌC: Drive trước, rồi các bản seed (zip)."""
    roots = []
    r = ckpt_root(scene)
    if r is not None and r.exists(): roots.append(r)
    for p in sorted(SEED_DIR.glob(f"**/ckpt/{scene}")) if SEED_DIR.exists() else []:
        if p.is_dir(): roots.append(p)
    # thư mục bạn upload thẳng lên /content bằng "Upload to Colab" (vd /content/ckpt-2026.../ckpt/<scene>/ hoặc /content/Data/ckpt-.../ckpt/<scene>/)
    for pat in (f"*/ckpt/{scene}", f"*/*/ckpt/{scene}"):
        for p in sorted(Path("/content").glob(pat)):
            if p.is_dir() and p not in roots and p.relative_to(Path("/content")).parts[0] not in ("drive", "ckpt_seed", "seed_zips"): roots.append(p)
    return roots

def ckpt_save_zip(scene, name, subs):
    root = ckpt_root(scene)
    if root is None: return
    ensure_drive(); root.mkdir(parents=True, exist_ok=True)
    base = data_dir(scene); tmp = root / (name + ".zip.tmp"); dst = root / (name + ".zip")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:           # ảnh jpg đã nén sẵn → không nén lại cho nhanh
        for sub in subs:
            for p in sorted((base / sub).rglob("*")):
                if p.is_file(): zf.write(p, str(p.relative_to(base)))
    tmp.replace(dst)                                                    # ghi xong mới đổi tên → không bao giờ để lại zip dở
    print(f"  💾 checkpoint [{scene}] {name}: {dst.stat().st_size / 1e6:.0f} MB → {dst}")

def ckpt_restore_zip(scene, name):
    if ckpt_root(scene) is None: return False
    ensure_drive(); seed_ckpt_zips()
    src = next((r / (name + ".zip") for r in ckpt_roots(scene) if (r / (name + ".zip")).exists()), None)
    if src is None: return False
    try:
        with zipfile.ZipFile(src) as zf:
            assert zf.testzip() is None
            zf.extractall(data_dir(scene))
    except Exception as e:
        print(f"  ⚠️ checkpoint {src} hỏng ({e}) → bỏ qua"); return False
    print(f"  ♻️ khôi phục [{scene}] {name} từ Drive")
    return True

def ckpt_sync_depth(scene):
    root = ckpt_root(scene)
    if root is None: return
    ensure_drive(); (root / "depth").mkdir(parents=True, exist_ok=True)
    for p in (data_dir(scene) / "depth").glob("*_pred.npy"):
        if not (root / "depth" / p.name).exists(): shutil.copy2(p, root / "depth" / p.name)

def ckpt_save_gs(scene, final):
    root = ckpt_root(scene)
    if root is None: return
    ensure_drive(); root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(final, root / "gs_point_cloud.ply")
    dp = data_dir(scene) / "sparse" / "0" / "depth_params.json"
    if dp.exists(): shutil.copy2(dp, root / "gs_depth_params.json")
    print(f"  💾 checkpoint [{scene}] 3DGS point cloud → {root}")

def ckpt_restore_gs(scene, final):
    if ckpt_root(scene) is None: return False
    ensure_drive(); seed_ckpt_zips()
    rr = next((r for r in ckpt_roots(scene) if (r / "gs_point_cloud.ply").exists()), None)
    if rr is None: return False
    src = rr / "gs_point_cloud.ply"
    final.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, final)
    dp = rr / "gs_depth_params.json"
    if dp.exists():
        (data_dir(scene) / "sparse" / "0").mkdir(parents=True, exist_ok=True)
        shutil.copy2(dp, data_dir(scene) / "sparse" / "0" / "depth_params.json")
    print(f"  ♻️ khôi phục [{scene}] 3DGS point cloud từ Drive")
    return True

# ===== zip kết quả cuối <scene>_prep.zip =====
def prep_zip_names(scene):
    p = max(1, int(ZIP_PARTS))
    return [f"{scene}_prep" + (f"_part{k + 1}of{p}" if p > 1 else "") + ".zip" for k in range(p)]

def verify_zip(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return zf.testzip() is None and len(zf.namelist()) > 0
    except Exception:
        return False

def prep_done_on_drive(scene):
    if not DRIVE_DIR.strip(): return False
    ensure_drive()
    return all(verify_zip(Path(DRIVE_DIR) / n) for n in prep_zip_names(scene))

def download_ckpt_bundle(scene):
    """Đóng gói checkpoint ĐẦY ĐỦ (colmap.zip + toàn bộ depth + 3DGS nếu có, gộp cả phần 'hạt giống' và phần mới) rồi tải về máy.
    Cấu trúc ckpt/<scene>/... giống zip bạn đã upload → dùng lại làm CKPT_SEED lần sau."""
    roots = ckpt_roots(scene)
    if not roots: return
    depth = {}
    for r in reversed(roots):                                              # root đầu (mới nhất) ghi đè
        for p in (r / "depth").glob("*_pred.npy"): depth[p.name] = p
    for p in (data_dir(scene) / "depth").glob("*_pred.npy"): depth.setdefault(p.name, p)
    single = {}
    for name in ("colmap.zip", "gs_point_cloud.ply", "gs_depth_params.json"):
        for r in roots:
            if (r / name).exists(): single[name] = r / name; break
    out = Path("/content") / f"ckpt_{scene}.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as zf:
        for name, p in single.items(): zf.write(p, f"ckpt/{scene}/{name}")
        for name, p in sorted(depth.items()): zf.write(p, f"ckpt/{scene}/depth/{name}")
    print(f"  📦 {out.name}: {out.stat().st_size / 1e6:.0f} MB ({len(depth)} depth, {list(single)}) ")
    try_download(out)


In [10]:
def data_dir(scene): return REPO / "data" / scene

def step_colmap(scene):
    d = data_dir(scene)
    def _done(): return (d / "sparse" / "0" / "points3D.bin").exists() and (d / "images").exists()
    if not _done():
        ckpt_restore_zip(scene, "colmap")
    if _done():
        print(f"[{scene}] COLMAP: đã có sparse/0 → bỏ qua")
        if ckpt_root(scene) is not None and not any((r / "colmap.zip").exists() for r in ckpt_roots(scene)):
            ckpt_save_zip(scene, "colmap", ["input", "images", "sparse"])
        return
    n = stage_images(scene, LERF_SCENES[scene])
    print(f"[{scene}] COLMAP trên {n} ảnh …")
    env = {**os.environ, "QT_QPA_PLATFORM": "offscreen"}
    # convert.py của tác giả dùng os.system → cần python có sẵn module; script chỉ dùng thư viện chuẩn
    sh(f"{VPY} convert.py -s data/{scene} --camera {CAMERA_MODEL}" + ("" if USE_COLMAP_GPU else " --no_gpu"), cwd=REPO, env=env)
    assert (d / "sparse" / "0" / "points3D.bin").exists(), f"[{scene}] COLMAP không tạo ra sparse/0/points3D.bin (ghép ảnh thất bại?)"
    ckpt_save_zip(scene, "colmap", ["input", "images", "sparse"])

CHECK_SFM = r"""
import sys, json, os
from PIL import Image
import pycolmap
scene = sys.argv[1]; root = f"data/{scene}"
rec = pycolmap.Reconstruction(f"{root}/sparse/0")
n_in = len([f for f in os.listdir(f"{root}/input")])
models = sorted({c.model.name for c in rec.cameras.values()})
bad = []
for img in rec.images.values():
    p = f"{root}/images/{img.name}"
    if not os.path.exists(p):                      # sau bước hoàn thiện ảnh đã được đổi tên .jpg → .JPEG
        p = f"{root}/images/{img.name.rsplit('.', 1)[0]}.JPEG"
    iw, ih = Image.open(p).size
    ow, oh = Image.open(f"{root}/input/{img.name}").size
    if (iw, ih) != (img.camera.width, img.camera.height) or (iw, ih) != (ow, oh):
        bad.append((img.name, (iw, ih), (ow, oh), (img.camera.width, img.camera.height)))
print(json.dumps({"input_images": n_in, "registered": len(rec.images), "points3D": len(rec.points3D), "models": models, "size_mismatch": bad[:5], "n_mismatch": len(bad)}))
"""

def check_sfm(scene):
    (REPO / "check_sfm_tmp.py").write_text(CHECK_SFM, encoding="utf-8")
    out = subprocess.run([str(VPY), "check_sfm_tmp.py", scene], cwd=REPO, env=ENV, capture_output=True, text=True)
    (REPO / "check_sfm_tmp.py").unlink()
    if out.returncode != 0:
        raise RuntimeError(f"[{scene}] kiểm tra COLMAP lỗi:\n{out.stderr[-1500:]}")
    r = json.loads(out.stdout.strip().splitlines()[-1])
    print(f"[{scene}] COLMAP: đăng ký {r['registered']}/{r['input_images']} ảnh, {r['points3D']} điểm 3D, camera {r['models']}")
    assert r["models"] == ["PINHOLE"] or CAMERA_MODEL != "PINHOLE", f"camera phải là PINHOLE, đang là {r['models']}"
    assert r["n_mismatch"] == 0, (f"Ảnh sau COLMAP khác kích thước ảnh gốc ở {r['n_mismatch']} frame (vd {r['size_mismatch']}): "
                                  "mask SAM 2 đã chạy sẽ không còn khớp → chạy lại SAM 2 trên data/<scene>/images, hoặc dùng CAMERA_MODEL=PINHOLE.")
    if r["registered"] < r["input_images"]:
        print(f"  ⚠️ {r['input_images'] - r['registered']} ảnh không đăng ký được; chúng sẽ không có depth/mask propagation.")
    return r

In [11]:
def _link(src, dst):
    try: os.symlink(Path(src).resolve(), dst)
    except OSError: shutil.copy2(src, dst)          # máy không cho symlink (vd Windows) → chép file

def step_murre(scene):
    d = data_dir(scene)
    out_dir = d / "murre"
    depth_dir = d / "depth"
    n_img = len([p for p in (d / "images").iterdir() if p.is_file()])      # undistorter chỉ xuất ảnh đã đăng ký
    if depth_dir.exists() and n_img > 0 and len(list(depth_dir.glob("*_pred.npy"))) >= n_img:
        print(f"[{scene}] Murre: đã có đủ {n_img} depth → bỏ qua"); ckpt_sync_depth(scene); return
    menv = {**os.environ, "PYTHONUNBUFFERED": "1"}
    print(f"[{scene}] Murre bước 1: tách depth thưa từ COLMAP")
    sh(f"{VPY_M} sfm_depth/get_sfm_depth.py --input_sfm_dir {d}/sparse/0 --output_sfm_dir {out_dir} --processing_res {MURRE_PROCESSING_RES}",
       cwd=MURRE, env=menv)
    # kiểm tra: depth COLMAP có vượt max_depth không (Murre cắt ở max_depth)
    import numpy as np
    zs = []
    for f in sorted((out_dir / "sparse_depth").glob("*.npz"))[::10]:
        a = np.load(f)["arr_0"][..., 0]
        zs.append(a[a > 0])
    zs = np.concatenate(zs) if zs else np.zeros(0)
    if zs.size:
        p50, p99 = np.percentile(zs, [50, 99])
        print(f"  depth SfM (đơn vị COLMAP): trung vị {p50:.2f}, p99 {p99:.2f}, max {zs.max():.2f} | max_depth Murre = {MURRE_MAX_DEPTH}")
        if p99 > 0.9 * MURRE_MAX_DEPTH:
            print(f"  ⚠️ p99 của depth SfM sát/vượt MURRE_MAX_DEPTH={MURRE_MAX_DEPTH}: depth sẽ bị cắt. Tăng MURRE_MAX_DEPTH rồi xoá {out_dir} và chạy lại.")
    print(f"[{scene}] Murre bước 2: ước lượng depth")
    out_npy = out_dir / "depth_npy"; out_npy.mkdir(parents=True, exist_ok=True)
    root = ckpt_root(scene)
    if root is not None:
        ensure_drive(); (root / "depth").mkdir(parents=True, exist_ok=True)
        n0 = 0
        seed_ckpt_zips()
        for rr in ckpt_roots(scene):
            for p in (rr / "depth").glob("*_pred.npy"):
                if not (out_npy / p.name).exists():
                    shutil.copy2(p, out_npy / p.name); n0 += 1
        if n0: print(f"  ♻️ khôi phục {n0} depth từ Drive")
    imgs = {p.stem: p for p in (d / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    todo = [k for k in sorted(imgs) if not (out_npy / f"{k}_pred.npy").exists()]
    print(f"  còn {len(todo)}/{len(imgs)} ảnh cần ước lượng depth" + (f" (lưu checkpoint mỗi {MURRE_CKPT_EVERY} ảnh)" if root is not None else ""))
    step = max(1, int(MURRE_CKPT_EVERY)) if root is not None else max(1, len(todo))
    for k0 in range(0, len(todo), step):
        chunk = todo[k0:k0 + step]
        t_rgb, t_sd = out_dir / "chunk_rgb", out_dir / "chunk_sdpt"          # Murre ghép ảnh với depth thưa theo tên → chỉ đưa nhóm cần làm
        for t in (t_rgb, t_sd):
            shutil.rmtree(t, ignore_errors=True); t.mkdir(parents=True)
        for k in chunk:
            _link(imgs[k], t_rgb / imgs[k].name)
            sd = out_dir / "sparse_depth" / f"{k}.npz"
            if sd.exists(): _link(sd, t_sd / sd.name)
        cmd = (f"{VPY_M} run.py --checkpoint {MURRE_CKPT} --input_rgb_dir {t_rgb} --input_sdpt_dir {t_sd} "
               f"--output_dir {out_dir} --denoise_steps {MURRE_DENOISE_STEPS} --ensemble_size {MURRE_ENSEMBLE_SIZE} "
               f"--processing_res {MURRE_PROCESSING_RES} --max_depth {MURRE_MAX_DEPTH}")
        if str(MURRE_ERR_THR).strip(): cmd += f" --err_thr {MURRE_ERR_THR}"
        if str(MURRE_NVIEWS_THR).strip(): cmd += f" --nviews_thr {MURRE_NVIEWS_THR}"
        sh(cmd, cwd=MURRE, env=menv)
        if root is not None:
            for k in chunk:
                p = out_npy / f"{k}_pred.npy"
                if p.exists(): shutil.copy2(p, root / "depth" / p.name)
            print(f"  💾 checkpoint depth: {len(list((root / 'depth').glob('*_pred.npy')))}/{len(imgs)} đã lưu trong checkpoint")
    shutil.rmtree(out_dir / "chunk_rgb", ignore_errors=True); shutil.rmtree(out_dir / "chunk_sdpt", ignore_errors=True)
    npys = sorted((out_dir / "depth_npy").glob("*_pred.npy"))
    assert npys, f"[{scene}] Murre không tạo ra depth_npy/*_pred.npy"
    depth_dir.mkdir(exist_ok=True)
    for f in npys:
        shutil.copy2(f, depth_dir / f.name)
    print(f"[{scene}] Murre: {len(npys)} depth → data/{scene}/depth/")

FIX_DEPTH_PY = r"""
import sys, glob, os, json
import numpy as np, cv2, pycolmap
scene = sys.argv[1]; root = f"data/{scene}"
rec = pycolmap.Reconstruction(f"{root}/sparse/0")
size = {img.name.rsplit('.', 1)[0]: (img.camera.height, img.camera.width) for img in rec.images.values()}
fixed, missing, stats = 0, [], []
for stem, (H, W) in size.items():
    f = f"{root}/depth/{stem}_pred.npy"
    if not os.path.exists(f):
        missing.append(stem); continue
    d = np.load(f)
    if d.shape != (H, W):
        d = cv2.resize(d.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)   # chỉ khi Murre trả kích thước khác ảnh
        np.save(f, d.astype(np.float32)); fixed += 1
    stats.append((float(np.nanmin(d)), float(np.nanmax(d)), float((d > 0).mean())))
st = np.array(stats) if stats else np.zeros((1, 3))
print(json.dumps({"registered": len(size), "missing": missing[:5], "n_missing": len(missing), "resized": fixed,
                  "depth_min": float(st[:, 0].min()), "depth_max": float(st[:, 1].max()), "frac_positive_min": float(st[:, 2].min())}))
"""

def check_depth(scene):
    (REPO / "fix_depth_tmp.py").write_text(FIX_DEPTH_PY, encoding="utf-8")
    out = subprocess.run([str(VPY), "fix_depth_tmp.py", scene], cwd=REPO, env=ENV, capture_output=True, text=True)
    (REPO / "fix_depth_tmp.py").unlink()
    if out.returncode != 0:
        raise RuntimeError(f"[{scene}] kiểm tra depth lỗi:\n{out.stderr[-1500:]}")
    r = json.loads(out.stdout.strip().splitlines()[-1])
    print(f"[{scene}] depth: {r['registered'] - r['n_missing']}/{r['registered']} frame, khoảng giá trị [{r['depth_min']:.3f}, {r['depth_max']:.3f}], "
          f"tỉ lệ pixel dương tối thiểu {r['frac_positive_min']:.2f}")
    if r["resized"]:
        print(f"  ⚠️ {r['resized']} depth có kích thước khác ảnh → đã resize bilinear về (H,W) của camera (tác giả không nêu bước này).")
    assert r["n_missing"] == 0, f"[{scene}] thiếu depth ở {r['n_missing']} frame, vd {r['missing']}"
    return r

In [12]:
INVDEPTH_PY = r"""
import sys, glob, os
import numpy as np, cv2
src, dst = sys.argv[1], sys.argv[2]
os.makedirs(dst, exist_ok=True)
files = sorted(glob.glob(f"{src}/*_pred.npy"))
invs, p999 = {}, []
for f in files:
    d = np.load(f).astype(np.float32)
    inv = np.where(d > 1e-6, 1.0 / np.maximum(d, 1e-6), 0.0).astype(np.float32)
    invs[f] = inv
    p999.append(float(np.percentile(inv[inv > 0], 99.9)) if (inv > 0).any() else 0.0)
norm = max(max(p999), 1e-8)           # 1 hằng số chung cho cả scene → tỉ lệ giữa các ảnh được giữ nguyên
for f, inv in invs.items():
    stem = os.path.basename(f)[: -len("_pred.npy")]
    cv2.imwrite(f"{dst}/{stem}.png", np.round(np.clip(inv / norm, 0, 1) * 65535.0).astype(np.uint16))
print(f"{len(files)} inverse-depth PNG, hằng số chuẩn hoá {norm:.4f}")
"""

def step_gs(scene):
    d = data_dir(scene)
    model = REPO / "output" / f"{scene}_dense"
    final = model / "point_cloud" / f"iteration_{GS_ITERATIONS}" / "point_cloud.ply"
    if not final.exists():
        ckpt_restore_gs(scene, final)
    if final.exists():
        print(f"[{scene}] 3DGS: đã có {final.name} → bỏ qua")
        if ckpt_root(scene) is not None and not any((r / "gs_point_cloud.ply").exists() for r in ckpt_roots(scene)):
            ckpt_save_gs(scene, final)
        return final
    (REPO / "invdepth_tmp.py").write_text(INVDEPTH_PY, encoding="utf-8")
    sh(f"{VPY} invdepth_tmp.py data/{scene}/depth data/{scene}/depths", cwd=REPO, env=ENV)
    (REPO / "invdepth_tmp.py").unlink()
    # utils/make_depth_scale.py của tác giả: khớp scale/offset inverse-depth từng ảnh với điểm COLMAP → sparse/0/depth_params.json
    sh(f"{VPY} utils/make_depth_scale.py --base_dir data/{scene} --depths_dir data/{scene}/depths", cwd=REPO, env=ENV)
    assert (d / "sparse" / "0" / "depth_params.json").exists()
    print(f"[{scene}] 3DGS có depth regularization, {GS_ITERATIONS} vòng …")
    sh(f"{VPY} train.py -s data/{scene} -d depths -m output/{scene}_dense --iterations {GS_ITERATIONS} "
       f"--test_iterations {GS_ITERATIONS} --save_iterations {GS_ITERATIONS} --disable_viewer", cwd=REPO, env=ENV)
    assert final.exists(), f"[{scene}] train.py không tạo {final}"
    ckpt_save_gs(scene, final)
    return final

def step_finalize(scene, ply):
    d = data_dir(scene)
    sp = d / "sparse" / "0"
    if (sp / "points3D_sfm.ply").exists():
        print(f"[{scene}] hoàn thiện: đã làm → bỏ qua"); return
    if (sp / "points3D.ply").exists():
        shutil.move(sp / "points3D.ply", sp / "points3D_sfm.ply")       # giữ lại bản điểm SfM gốc
    shutil.copy2(ply, sp / "points3D.ply")
    n = 0
    for p in sorted((d / "images").glob("*.jpg")):                       # đúng thao tác của auto_seg.py: jpg → JPEG
        p.rename(p.with_suffix(".JPEG")); n += 1
    print(f"[{scene}] points3D.ply = point cloud 3DGS ({sp / 'points3D.ply'}), đổi tên {n} ảnh .jpg → .JPEG")

## 6. Đóng gói và tải về

In [13]:
import numpy as np

def zip_prep(scene, parts=1):
    base = data_dir(scene)
    parts = max(1, int(parts))
    zdir = Path("/content") if Path("/content").is_dir() else Path(".")
    files = []
    for sub in ("images", "sparse/0", "depth"):
        files += [f for f in (base / sub).rglob("*") if f.is_file()]
    files = sorted(files)
    zips = [zdir / (f"{scene}_prep" + (f"_part{k + 1}of{parts}" if parts > 1 else "") + ".zip") for k in range(parts)]
    hs = [zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED, compresslevel=1) for z in zips]
    try:
        for i, f in enumerate(files):
            hs[i * parts // len(files)].write(f, arcname=str(Path(scene) / f.relative_to(base)))
    finally:
        for h in hs:
            h.close()
    for z in zips:
        mb = z.stat().st_size / 1e6
        print(f"  {z.name}: {mb:.0f} MB" + ("  ⚠️ lớn, nên tăng ZIP_PARTS" if mb > 1500 else ""))
    return zips

def download_prep(scene):
    for z in zip_prep(scene, ZIP_PARTS):
        try_download(z)

## 7. Chạy toàn bộ

Với mỗi scene: COLMAP → Murre → 3DGS → hoàn thiện → kiểm tra → (tải về). Mất nhiều thời gian (COLMAP CPU là bước lâu nhất); nếu Colab ngắt, chạy lại từ ô 2 — các bước đã xong sẽ được bỏ qua (nếu runtime còn dữ liệu), và với `after_each_scene` scene xong trước đó đã nằm trong máy bạn.

In [14]:
import shutil, traceback, zipfile
from google.colab import drive

# Checkpoint: lưu <scene>_prep.zip lên Drive NGAY SAU MỖI SCENE (để Colab ngắt giữa chừng cũng không mất scene đã xong)
DRIVE_OUT = Path(DRIVE_DIR) if DRIVE_DIR.strip() else None   # DRIVE_DIR trống = không lưu lên Drive
if DRIVE_OUT:
    ensure_drive()                             # mount ngay từ đầu để bấm Allow khi còn ở máy, không đợi hàng giờ
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def save_scene_to_drive(scene):
    for z in zip_prep(scene, ZIP_PARTS):
        dst = DRIVE_OUT / z.name
        shutil.copy2(z, dst)
        assert verify_zip(dst), f"{dst} hỏng"
        assert dst.stat().st_size == z.stat().st_size, f"{dst} sai dung lượng"
        print(f"  ✅ [{scene}] đã lưu {dst} ({dst.stat().st_size / 1e6:.0f} MB)")

REPORT, FAILED, SKIPPED = [], {}, []
t_all = time.time()
for scene in LERF_SCENES:
    print(f"\n=================== {scene} ===================")
    t0 = time.time()
    if DRIVE_OUT and prep_done_on_drive(scene):
        print(f"[{scene}] đã có {prep_zip_names(scene)} hợp lệ trên Drive → bỏ qua scene này")
        SKIPPED.append(scene); continue
    try:                                       # một scene lỗi không làm chết scene còn lại
        step_colmap(scene)
        sfm = check_sfm(scene)
        step_murre(scene)
        dep = check_depth(scene)
        if AUTO_DOWNLOAD_CKPT:
            download_ckpt_bundle(scene)        # xong hết depth (vd 305 ảnh) → tải checkpoint về máy ngay, phòng khi phiên bị ngắt ở bước 3DGS
        ply = step_gs(scene)
        step_finalize(scene, ply)
        REPORT.append({"scene": scene, "ảnh đăng ký COLMAP": sfm["registered"], "điểm SfM": sfm["points3D"],
                       "depth min": round(dep["depth_min"], 3), "depth max": round(dep["depth_max"], 3),
                       "phút": round((time.time() - t0) / 60, 1)})
        if DRIVE_OUT:
            save_scene_to_drive(scene)
        if DOWNLOAD_WHEN == "after_each_scene":
            download_prep(scene)
    except Exception:
        FAILED[scene] = traceback.format_exc()
        print(f"\n❌ [{scene}] LỖI — bỏ qua, chạy tiếp scene sau. 25 dòng cuối:\n" + "\n".join(FAILED[scene].splitlines()[-25:]))

import pandas as pd
if REPORT:
    display(pd.DataFrame(REPORT).set_index("scene"))
print(f"Tổng {(time.time() - t_all) / 60:.1f} phút | xong: {[r['scene'] for r in REPORT]} | đã có sẵn trên Drive: {SKIPPED} | lỗi: {list(FAILED)}")
if DOWNLOAD_WHEN == "at_end":
    for scene in [r["scene"] for r in REPORT]:
        download_prep(scene)



=================== figurines ===================
$ gdown 1cPFOvfkLvqKY09yo098ndka_PFfnhYzq -O /content/seed_zips/
Downloading...
From (original): https://drive.google.com/uc?id=1cPFOvfkLvqKY09yo098ndka_PFfnhYzq
From (redirected): https://drive.google.com/uc?id=1cPFOvfkLvqKY09yo098ndka_PFfnhYzq&confirm=t&uuid=ab425c3a-2710-4bd7-a205-4fc6d1cb8c10
To: /content/seed_zips/ckpt-20261008T085313Z-1-001.zip
  0%|          | 0.00/439M [00:00<?, ?B/s]
  0%|          | 1.05M/439M [00:00<00:47, 9.32MB/s]
  1%|          | 4.72M/439M [00:00<00:29, 14.7MB/s]
  3%|▎         | 11.0M/439M [00:00<00:14, 29.6MB/s]
  3%|▎         | 14.7M/439M [00:00<00:22, 19.2MB/s]
  5%|▌         | 23.6M/439M [00:01<00:16, 24.9MB/s]
  7%|▋         | 29.4M/439M [00:01<00:13, 30.3MB/s]
  8%|▊         | 34.1M/439M [00:01<00:15, 26.9MB/s]
  9%|▊         | 38.3M/439M [00:01<00:14, 27.2MB/s]
 10%|█         | 44.6M/439M [00:01<00:13, 29.7MB/s]
 12%|█▏        | 50.9M/439M [00:01<00:11, 32.5MB/s]
 13%|█▎        | 57.1M/439M [00:0

: 

In [ ]:
#@title 8. Đóng gói kết quả để lấy về máy { display-mode: "form" }
for scene in LERF_SCENES:
    d = data_dir(scene)
    if not ((d / "sparse" / "0" / "points3D.bin").exists() and (d / "depth").exists()):
        print(f"[{scene}] ❌ chưa có kết quả trong máy → bỏ qua"); continue
    print(f"[{scene}] đóng gói:")
    for z in zip_prep(scene, ZIP_PARTS):
        try_download(z)
    download_ckpt_bundle(scene)
print("\n✅ Xong. KHÔNG tắt phiên cho tới khi bạn đã chép các file ở trên về máy (Colab xoá sạch /content khi hết phiên).")


## Sau notebook này

Mỗi scene có `data/<scene>/{images/*.JPEG, sparse/0/, depth/*_pred.npy}`. Đưa `<scene>_prep.zip` vào notebook **lan truyền mask** (`PREP_GDRIVE_IDS` hoặc upload vào `PREP_ZIP_DIR`), cùng với 2 zip mask SAM 2 của bạn.

`depth_params.json`, `points3D_sfm.ply` (điểm SfM gốc) nằm trong `sparse/0/` để tiện đối chiếu.